<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# MarketPulse — Analytics Marketing Digital
## Notebook 3 — SQL Analytics : Performance & Attribution
### ✅ VERSION CORRIGÉE

> Les blocs `MÉTHODE` expliquent les choix techniques et les patterns généralisables.  
> Les blocs `INTERPRÉTATION` lisent les résultats avec les chiffres réels.  
> Les blocs `MÉTIER` font le lien entre le chiffre et la décision business.

| | |
|---|---|
| **Prérequis** | NB1 et NB2 complétés — `marketpulse_analytics.csv` disponible |
| **Outils** | DuckDB (JupySQL), pandas, matplotlib |
| **Durée estimée** | 3h à 4h |

> **Objectif :** Répondre aux questions business de MarketPulse avec du SQL analytique avancé. Chaque requête combine `RANK()`, `LAG()`, ou `NTILE()` avec une interprétation métier directement actionnable par un account manager. On travaille sur deux niveaux de granularité : **campagne** (depuis `marketpulse_analytics.csv`) et **quotidien** (depuis `performances_daily.csv` pour les heatmaps et tendances intra-campagne).

---
## 0. Mise en place

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import duckdb, os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#F9F9F8',
                      'axes.grid':True,'grid.alpha':0.35,'font.size':11})

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/marketpulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 {"Colab" if IN_COLAB else "Local"} — {SAVE_PATH}')

---
## 1. Connexion DuckDB et chargement des tables

### MÉTHODE — Deux niveaux de granularité
NB3 utilise **deux granularités** selon la question posée :
- `marketpulse_analytics` — granularité **campagne** (354 lignes). Utilisée pour les KPIs, rankings, segmentations NTILE et tendances mensuelles
- `performances_daily` — granularité **campagne × jour** (8 861 lignes). Utilisée pour les heatmaps et l'analyse intra-campagne
- `contacts_crm` — granularité **contact** (9 500 lignes). Utilisée pour le calcul de LTV par canal d'acquisition

> **Note :** `marketpulse_analytics.csv` est le fichier exporté par le NB2. Si ce fichier n'est pas disponible localement (première exécution), il peut être chargé depuis le dépôt GitHub.

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/marketpulse/data/'

conn = duckdb.connect()

# Essai chargement depuis SAVE_PATH (NB2), sinon GitHub
analytics_path = f'{SAVE_PATH}marketpulse_analytics.csv'
if os.path.exists(analytics_path):
    conn.execute(f"CREATE TABLE marketpulse_analytics AS SELECT * FROM read_csv_auto('{analytics_path}', timestampformat='%Y-%m-%d')")
    print('marketpulse_analytics chargé depuis SAVE_PATH (NB2)')
else:
    conn.execute(f"CREATE TABLE marketpulse_analytics AS SELECT * FROM read_csv_auto('{BASE_URL}marketpulse_analytics.csv', timestampformat='%Y-%m-%d')")
    print('marketpulse_analytics chargé depuis GitHub')

conn.execute(f"""
    CREATE TABLE performances_daily AS
        SELECT * FROM read_csv_auto('{BASE_URL}performances_daily.csv', timestampformat='%Y-%m-%d');
    CREATE TABLE campagnes AS
        SELECT * FROM read_csv_auto('{BASE_URL}campagnes.csv', timestampformat='%Y-%m-%d');
    CREATE TABLE clients_agence AS
        SELECT * FROM read_csv_auto('{BASE_URL}clients_agence.csv', timestampformat='%Y-%m-%d');
    CREATE TABLE contacts_crm AS
        SELECT * FROM read_csv_auto('{BASE_URL}contacts_crm.csv', timestampformat='%Y-%m-%d');
    CREATE TABLE audiences AS
        SELECT * FROM read_csv_auto('{BASE_URL}audiences.csv');
""")

# Vérification
for t in ['marketpulse_analytics','performances_daily','campagnes','clients_agence','contacts_crm']:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t:<30} {n:>8,} lignes')

# Recalcul des colonnes nettoyées dans performances_daily
conn.execute("""
    ALTER TABLE performances_daily ADD COLUMN IF NOT EXISTS ctr_recalc DOUBLE;
    ALTER TABLE performances_daily ADD COLUMN IF NOT EXISTS roas_recalc DOUBLE;
    UPDATE performances_daily
    SET ctr_recalc  = CASE WHEN impressions > 0 THEN ROUND(clics * 1.0 / impressions, 4) ELSE NULL END,
        roas_recalc = CASE WHEN depenses_fcfa > 0 AND revenu_genere_fcfa >= 0
                           THEN ROUND(revenu_genere_fcfa * 1.0 / depenses_fcfa, 3) ELSE NULL END
""")
print('\nColonnes recalculées ajoutées ✅')

In [ ]:
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')

---
## 2. Vue consolidée — KPIs depuis `marketpulse_analytics`

### MÉTHODE
Première requête sur la table analytique consolidée du NB2. On utilise `marketpulse_analytics` plutôt que `performances_daily` car toutes les corrections et recalculs ont déjà été appliqués — c'est la **source de vérité unique** pour les analyses campagne-niveau.

In [ ]:
%%sql df_kpi_global <<
SELECT
    COUNT(*)                                                              AS nb_campagnes,
    COUNT(DISTINCT client_id)                                             AS nb_clients,
    COUNT(DISTINCT canal)                                                 AS nb_canaux,
    -- Performance
    ROUND(AVG(roas_total), 3)                                             AS roas_moyen,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY roas_total), 3)     AS roas_median,
    ROUND(AVG(ctr_total) * 100, 2)                                        AS ctr_pct_moyen,
    ROUND(AVG(cpa_total), 0)                                              AS cpa_moyen_fcfa,
    ROUND(AVG(cpc_total), 0)                                              AS cpc_moyen_fcfa,
    -- Sous-performance
    SUM(campagne_sous_performe)                                           AS nb_sous_perf,
    ROUND(SUM(campagne_sous_performe) * 100.0 / COUNT(*), 1)              AS pct_sous_perf,
    -- Signaux précoces
    ROUND(AVG(roas_j3), 3)                                                AS roas_j3_moyen,
    ROUND(AVG(ctr_j3) * 100, 2)                                           AS ctr_j3_pct_moyen,
    -- Durée
    ROUND(AVG(duree_jours), 1)                                            AS duree_moy_jours
FROM marketpulse_analytics

> **INTERPRÉTATION — Vue consolidée depuis `marketpulse_analytics` :**
>
> | KPI | Valeur | Lecture |
> |---|---|---|
> | ROAS moyen | **3.31×** | Médiane ~2.8× — quelques campagnes très performantes tirent la moyenne vers le haut |
> | CTR moyen | **2.86%** | ~1 clic pour 35 impressions — hétérogène selon le canal |
> | CPA moyen | **~7 287 FCFA** | Toutes conversions confondues — Lead < Achat |
> | Campagnes sous-perf | **26.3%** (93/354) | Variable cible ML NB5 |
> | ROAS J3 moyen | **~2.1×** | Plus bas que le final — les campagnes s'améliorent généralement avec le temps |
> | Durée moyenne | **~25 jours** | Campagnes mensuelles récurrentes |
>
> **MÉTIER :** L'écart entre le ROAS moyen (3.31×) et le ROAS J3 moyen (~2.1×) confirme que les 3 premiers jours d'une campagne sont sa période la moins efficiente — l'algorithme de diffusion optimise progressivement le ciblage. C'est pourquoi le seuil de décision ML ne peut pas être ROAS J3 < 1.5 directement, mais doit intégrer cette trajectoire normale de montée en puissance.

---
## 3. Classement des canaux avec RANK()

### MÉTHODE
Un seul passage sur la table suffit pour calculer trois classements simultanés : ROAS, coût d'acquisition et fiabilité. La CTE isole le calcul des agrégats, la requête externe ajoute les rangs avec `RANK() OVER (ORDER BY ...)`. Ce pattern évite les jointures multiples et garantit que tous les rangs sont calculés sur le **même périmètre** de données.

In [ ]:
%%sql df_rank_canal <<
WITH perf_canal AS (
    SELECT
        canal,
        COUNT(*)                                                          AS nb_campagnes,
        ROUND(AVG(roas_total), 2)                                         AS roas_moyen,
        ROUND(AVG(roas_j3), 2)                                            AS roas_j3_moyen,
        ROUND(AVG(ctr_total) * 100, 2)                                    AS ctr_pct,
        ROUND(AVG(cpc_total), 0)                                          AS cpc_fcfa,
        ROUND(AVG(cpa_total), 0)                                          AS cpa_fcfa,
        ROUND(AVG(duree_jours), 1)                                        AS duree_moy_jours,
        SUM(campagne_sous_performe)                                       AS nb_sous_perf,
        ROUND(SUM(campagne_sous_performe) * 100.0 / COUNT(*), 1)          AS pct_sous_perf
    FROM marketpulse_analytics
    GROUP BY canal
)
SELECT
    canal,
    nb_campagnes,
    roas_moyen,
    roas_j3_moyen,
    ctr_pct,
    cpc_fcfa,
    cpa_fcfa,
    pct_sous_perf,
    duree_moy_jours,
    RANK() OVER (ORDER BY roas_moyen DESC)     AS rang_roas,
    RANK() OVER (ORDER BY cpc_fcfa ASC)        AS rang_cout,
    RANK() OVER (ORDER BY cpa_fcfa ASC)        AS rang_acquisition,
    RANK() OVER (ORDER BY pct_sous_perf ASC)   AS rang_fiabilite,
    -- Score composite : meilleure somme de rangs = meilleur canal global
    RANK() OVER (ORDER BY roas_moyen DESC) +
    RANK() OVER (ORDER BY cpc_fcfa ASC) +
    RANK() OVER (ORDER BY pct_sous_perf ASC)   AS score_composite
FROM perf_canal
ORDER BY rang_roas

> **INTERPRÉTATION — Classement multi-critères des canaux :**
>
> | Canal | ROAS | CPC (FCFA) | CPA (FCFA) | Sous-perf | Score composite |
> |---|---|---|---|---|---|
> | **Email** | **4.67×** | **46** | le plus bas | **0.0%** | 🏆 **3** (meilleur) |
> | **SMS** | 3.44× | **30** | bas | 30.0% | **5** |
> | **Google** | 2.92× | 226 | élevé | 33.3% | **9** |
> | **Facebook** | 2.51× | 153 | moyen | 39.2% | **11** |
>
> **Lecture du rang composite :** En combinant ROAS, CPC et fiabilité, Email est le canal le plus efficace toutes dimensions confondues. SMS est surprenamment fort : son CPC (30 FCFA) est le plus bas du portefeuille — le coût d'envoi d'un SMS est quasi fixe et très faible en Afrique de l'Ouest.
>
> **ROAS J3 vs ROAS final :** Le ratio `roas_j3_moyen / roas_moyen` mesure la vitesse de montée en puissance. Email a le ratio le plus élevé (peu d'amélioration entre J3 et la fin) — normal : une liste opt-in bien segmentée performe dès le début. Facebook a le ratio le plus bas — son algorithme met plus de temps à trouver les bonnes audiences.
>
> **MÉTIER :** Ce tableau est l'outil de décision d'allocation budgétaire mensuelle. Un client avec un budget limité doit investir en priorité sur Email > SMS. Google est justifié uniquement pour des produits à haute valeur (LTV > 50k FCFA) où le CPA élevé est absorbable. Facebook reste pertinent pour la notoriété et l'acquisition de nouvelles audiences, pas pour la conversion directe.

---
## 4. Performance croisée canal × objectif

### MÉTHODE
Le croisement canal × objectif révèle les **combinaisons gagnantes**. `RANK() OVER (PARTITION BY objectif_campagne ORDER BY roas_moyen DESC)` classe les canaux **au sein de chaque objectif** — ce qui donne une recommandation plus précise qu'un classement global.

In [ ]:
%%sql df_canal_objectif <<
WITH croise AS (
    SELECT
        canal,
        objectif_campagne,
        COUNT(*)                                                          AS nb_campagnes,
        ROUND(AVG(roas_total), 2)                                         AS roas_moyen,
        ROUND(AVG(ctr_total) * 100, 2)                                    AS ctr_pct,
        ROUND(SUM(campagne_sous_performe) * 100.0 / COUNT(*), 1)          AS pct_sous_perf
    FROM marketpulse_analytics
    GROUP BY canal, objectif_campagne
    HAVING COUNT(*) >= 3
)
SELECT
    objectif_campagne,
    canal,
    nb_campagnes,
    roas_moyen,
    ctr_pct,
    pct_sous_perf,
    RANK() OVER (PARTITION BY objectif_campagne ORDER BY roas_moyen DESC) AS rang_dans_objectif
FROM croise
ORDER BY objectif_campagne, rang_dans_objectif

> **INTERPRÉTATION — Classement canal par objectif :**
>
> Ce tableau répond à la question opérationnelle d'un account manager : *"Mon client veut faire de la Conversion — quel canal lui recommander ?"*
>
> Quelques patterns attendus :
> - **Conversion** → Email rang 1 systématiquement — les listes opt-in ont déjà fait l'étape de sélection
> - **Notoriété** → Facebook rang 1 — la portée et le coût faible par impression font de Facebook le canal idéal pour la notoriété, même si le ROAS en conversion est médiocre
> - **Retargeting** → Email et SMS dominent — relancer un client qui a déjà montré de l'intérêt fonctionne mieux sur les canaux directs
> - **Leads** → Google rang élevé — l'intention de recherche sur Google génère des leads plus qualifiés
>
> **MÉTIER :** La combinaison `canal × objectif` est la première variable à vérifier avant de lancer une campagne. Un Facebook Ads avec objectif Conversion a statistiquement 39%+ de chances de sous-performer — contre 0% pour un Email Conversion. Ce tableau est directement utilisable comme règle métier dans le système d'alerte NB5.

---
## 5. Tendance mensuelle avec LAG()

### MÉTHODE
`LAG(roas) OVER (ORDER BY mois)` retourne la valeur du mois précédent dans la même partition. La variation `roas - LAG(roas)` mesure l'accélération ou la dégradation mois par mois. On utilise `strftime(date_debut, '%Y-%m')` sur `marketpulse_analytics` — cela group par mois de lancement de campagne, pas par mois d'activité. C'est le choix le plus cohérent : une campagne lancée en novembre reste en novembre même si elle tourne jusqu'en décembre.

In [ ]:
%%sql df_lag_mensuel <<
WITH base AS (
    SELECT
        strftime(date_debut, '%Y-%m')                                      AS mois,
        COUNT(*)                                                           AS nb_campagnes,
        ROUND(AVG(roas_total), 3)                                          AS roas,
        ROUND(AVG(ctr_total) * 100, 2)                                     AS ctr_pct,
        ROUND(AVG(cpa_total), 0)                                           AS cpa_fcfa,
        ROUND(SUM(campagne_sous_performe) * 100.0 / COUNT(*), 1)           AS pct_sous_perf,
        ROUND(AVG(duree_jours), 0)                                         AS duree_moy
    FROM marketpulse_analytics
    GROUP BY strftime(date_debut, '%Y-%m')
)
SELECT
    mois,
    nb_campagnes,
    roas,
    LAG(roas)     OVER (ORDER BY mois)                                     AS roas_prev,
    ROUND(roas - LAG(roas) OVER (ORDER BY mois), 3)                        AS delta_roas,
    ctr_pct,
    ROUND(ctr_pct - LAG(ctr_pct) OVER (ORDER BY mois), 2)                 AS delta_ctr,
    cpa_fcfa,
    ROUND(cpa_fcfa - LAG(cpa_fcfa) OVER (ORDER BY mois), 0)               AS delta_cpa,
    pct_sous_perf,
    -- Signal d'alerte : deux mois consécutifs de baisse
    CASE
        WHEN roas < LAG(roas) OVER (ORDER BY mois)
         AND LAG(roas) OVER (ORDER BY mois) < LAG(roas, 2) OVER (ORDER BY mois)
        THEN '⚠️ Déclin 2 mois'
        WHEN roas < LAG(roas) OVER (ORDER BY mois)
        THEN '↘ Baisse'
        WHEN roas > LAG(roas) OVER (ORDER BY mois)
        THEN '↗ Hausse'
        ELSE '→ Stable'
    END                                                                    AS tendance
FROM base
ORDER BY mois

> **INTERPRÉTATION — Tendance mensuelle avec LAG() :**
>
> La colonne `tendance` identifie automatiquement les mois de déclin consécutif — signal d'alerte pour le pilotage de l'agence. Patterns observés :
> - **Hausse** en fin d'année (Q4) — budgets de fin d'année des clients, campagnes promotionnelles Noël/Nouvel An
> - **Baisse** en début d'année (Q1) — fatigue des audiences post-fêtes, budgets gelés en attente de validation
> - **Déclin 2 mois consécutifs** → signal critique pour l'agence : si observé sur 2+ clients simultanément, c'est souvent un problème systémique (saturation plateforme, concurrence saisonnière, créatifs vieillis)
>
> **Pattern de LAG à 2 périodes :** `LAG(roas, 2)` (deuxième argument = décalage de 2 périodes) compare avec le mois N-2 — permet de détecter les tendances baissières continues, pas juste les corrections ponctuelles.
>
> **MÉTIER :** La colonne `tendance` est le **KPI de monitoring mensuel** à afficher en header du dashboard Power BI. Deux mois consécutifs de baisse ROAS sur un client = réunion de révision stratégique obligatoire avec l'account manager.

---
## 6. Segmentation des campagnes avec NTILE(3)

### MÉTHODE
`NTILE(3) OVER (ORDER BY roas_total DESC)` divise les 354 campagnes en 3 tiers égaux (118 campagnes chacun). Contrairement à des seuils fixes (ROAS > 3.5 = Top), NTILE garantit des tiers toujours équilibrés quelle que soit la distribution. On calcule ensuite le profil de chaque tier pour identifier les patterns qui distinguent les meilleures campagnes des pires.

In [ ]:
%%sql df_ntile_camp <<
WITH tiers AS (
    SELECT
        campagne_id,
        client_id,
        canal,
        objectif_campagne,
        pays_cible,
        duree_jours,
        roas_total,
        roas_j3,
        ctr_total,
        ctr_j3,
        variation_ctr_j1_j3,
        campagne_sous_performe,
        est_lancement_weekend,
        NTILE(3) OVER (ORDER BY roas_total DESC)  AS tier_roas
    FROM marketpulse_analytics
    WHERE roas_total IS NOT NULL
)
SELECT
    tier_roas,
    CASE tier_roas
        WHEN 1 THEN 'Top performer'
        WHEN 2 THEN 'Intermédiaire'
        ELSE       'À optimiser'
    END                                                                    AS segment,
    COUNT(*)                                                               AS nb_campagnes,
    ROUND(MIN(roas_total), 2)                                              AS roas_min,
    ROUND(AVG(roas_total), 2)                                              AS roas_moyen,
    ROUND(MAX(roas_total), 2)                                              AS roas_max,
    ROUND(AVG(roas_j3), 2)                                                 AS roas_j3_moyen,
    ROUND(AVG(ctr_total) * 100, 2)                                         AS ctr_pct_moyen,
    ROUND(AVG(variation_ctr_j1_j3) * 100, 2)                              AS var_ctr_j3_pct,
    ROUND(AVG(duree_jours), 1)                                             AS duree_moy,
    ROUND(SUM(est_lancement_weekend) * 100.0 / COUNT(*), 1)               AS pct_weekend,
    ROUND(SUM(campagne_sous_performe) * 100.0 / COUNT(*), 1)              AS pct_sous_perf
FROM tiers
GROUP BY tier_roas
ORDER BY tier_roas

In [ ]:
%%sql df_ntile_canal <<
-- Composition des tiers par canal
WITH tiers AS (
    SELECT canal,
           NTILE(3) OVER (ORDER BY roas_total DESC) AS tier
    FROM marketpulse_analytics
    WHERE roas_total IS NOT NULL
)
SELECT
    canal,
    SUM(CASE WHEN tier = 1 THEN 1 ELSE 0 END)  AS top_performer,
    SUM(CASE WHEN tier = 2 THEN 1 ELSE 0 END)  AS intermediaire,
    SUM(CASE WHEN tier = 3 THEN 1 ELSE 0 END)  AS a_optimiser,
    COUNT(*)                                    AS total,
    ROUND(SUM(CASE WHEN tier = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_top
FROM tiers
GROUP BY canal
ORDER BY pct_top DESC

> **INTERPRÉTATION — Segmentation NTILE(3) :**
>
> **Profil des 3 tiers :**
>
> - **Top performer (Tier 1)** : ROAS > ~4.5×, ROAS J3 déjà élevé (~3.5×), variation CTR légèrement négative (saturation naturelle après une forte accroche initiale), durée plutôt courte (~15-18j). Ce sont les campagnes bien ciblées sur des audiences fraîches avec des offres claires
> - **Intermédiaire (Tier 2)** : ROAS autour de 3.0-3.5×, performances correctes mais sans excellence. La grande majorité des campagnes — le cœur du portefeuille
> - **À optimiser (Tier 3)** : ROAS < 1.5×, ROAS J3 déjà faible (~0.8×), variation CTR fortement négative (saturation rapide), durée plus longue. Ces campagnes auraient dû être stoppées ou ajustées dès J3-J5
>
> **Composition par canal :** La proportion de Top Performers est la plus forte sur Email (~50-60% dans le Tier 1) et la plus faible sur Facebook (~20-25%). Ce résultat confirme le classement RANK() de la requête précédente avec une granularité différente.
>
> **MÉTIER :** Les campagnes du Tier 3 (À optimiser) sont le **vivier prioritaire du NB5**. En production, le système d'alerte ML serait déclenché dès J3 pour identifier les futures campagnes Tier 3 avant qu'elles aient consommé 100% de leur budget sans atteindre leurs objectifs.

---
## 7. Heatmap ROAS : jour de semaine × canal

### MÉTHODE
On utilise `performances_daily` (granularité quotidienne) pour calculer le ROAS moyen par jour de la semaine et par canal. `strftime(date_perf, '%w')` retourne l'entier du jour (0=Dimanche en DuckDB/SQLite). On pivote ensuite le résultat en pandas avec `pivot_table()` pour construire la matrice 4 × 7 de la heatmap.

In [ ]:
%%sql df_heatmap_raw <<
WITH perf_clean AS (
    SELECT *
    FROM performances_daily
    WHERE clics          <= impressions
      AND revenu_genere_fcfa >= 0
      AND ctr            <= 0.20
      AND roas_recalc IS NOT NULL
)
SELECT
    canal,
    CASE CAST(strftime(date_perf, '%w') AS INTEGER)
        WHEN 1 THEN '01-Lun'
        WHEN 2 THEN '02-Mar'
        WHEN 3 THEN '03-Mer'
        WHEN 4 THEN '04-Jeu'
        WHEN 5 THEN '05-Ven'
        WHEN 6 THEN '06-Sam'
        WHEN 0 THEN '07-Dim'
    END                                                                    AS jour_semaine,
    ROUND(AVG(roas_recalc), 3)                                             AS roas_moyen,
    ROUND(AVG(ctr_recalc) * 100, 2)                                        AS ctr_pct,
    COUNT(*)                                                               AS nb_obs
FROM perf_clean
GROUP BY canal, CAST(strftime(date_perf, '%w') AS INTEGER)
ORDER BY canal, jour_semaine

In [ ]:
# Pivot pour la heatmap
df_hm_roas = df_heatmap_raw.pivot_table(
    index='canal', columns='jour_semaine', values='roas_moyen', aggfunc='mean'
)
df_hm_ctr = df_heatmap_raw.pivot_table(
    index='canal', columns='jour_semaine', values='ctr_pct', aggfunc='mean'
)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Heatmap ROAS
im1 = axes[0].imshow(df_hm_roas.values, aspect='auto',
                      cmap='RdYlGn', vmin=1.5, vmax=6.0)
axes[0].set_xticks(range(len(df_hm_roas.columns)))
axes[0].set_xticklabels([c.replace('0','').replace('-',' ') for c in df_hm_roas.columns],
                         fontsize=9)
axes[0].set_yticks(range(len(df_hm_roas.index)))
axes[0].set_yticklabels(df_hm_roas.index, fontsize=10)
for i in range(df_hm_roas.shape[0]):
    for j in range(df_hm_roas.shape[1]):
        v = df_hm_roas.iloc[i, j]
        if not pd.isna(v):
            axes[0].text(j, i, f'{v:.2f}', ha='center', va='center',
                          fontsize=8, color='black' if 2 < v < 5 else 'white')
plt.colorbar(im1, ax=axes[0], label='ROAS')
axes[0].set_title('ROAS moyen par canal × jour', fontweight='bold')
axes[0].grid(False)

# Heatmap CTR
im2 = axes[1].imshow(df_hm_ctr.values, aspect='auto',
                      cmap='Blues', vmin=1.0, vmax=5.0)
axes[1].set_xticks(range(len(df_hm_ctr.columns)))
axes[1].set_xticklabels([c.replace('0','').replace('-',' ') for c in df_hm_ctr.columns],
                         fontsize=9)
axes[1].set_yticks(range(len(df_hm_ctr.index)))
axes[1].set_yticklabels(df_hm_ctr.index, fontsize=10)
for i in range(df_hm_ctr.shape[0]):
    for j in range(df_hm_ctr.shape[1]):
        v = df_hm_ctr.iloc[i, j]
        if not pd.isna(v):
            axes[1].text(j, i, f'{v:.2f}%', ha='center', va='center',
                          fontsize=8, color='white' if v > 3.5 else 'black')
plt.colorbar(im2, ax=axes[1], label='CTR (%)')
axes[1].set_title('CTR moyen par canal × jour', fontweight='bold')
axes[1].grid(False)

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}marketpulse_heatmap_canal_jour.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}marketpulse_heatmap_canal_jour.png')

> **INTERPRÉTATION — Heatmap ROAS et CTR jour × canal :**
>
> **Email :** Performances homogènes 7j/7 — une campagne email envoyée le mardi ou le jeudi performe quasiment pareil. C'est attendu : le destinataire lit son email à n'importe quel moment de la semaine
>
> **SMS :** Légère supériorité le weekend — en Afrique de l'Ouest, le dimanche est un jour de forte consultation mobile (loisirs, famille). Les SMS promotionnels perçus comme moins intrusifs
>
> **Facebook :** ROAS plus élevé en milieu de semaine (mar-mer-jeu) — les audiences B2C sont plus actives en semaine, et les CPC sont souvent moins chers (moins de concurrence publicitaire)
>
> **Google :** Pic le mercredi et jeudi — l'intention de recherche commerciale est plus forte en milieu de semaine quand les gens planifient leurs achats
>
> **Pattern weekend :** Le samedi-dimanche montre généralement des CTR plus élevés mais des ROAS plus variables — plus de trafic mais qualité d'audience moins homogène
>
> **MÉTIER :** Ces patterns orientent la **stratégie de diffusion** : pour maximiser le ROAS, programmer les campagnes Facebook le mardi-jeudi, les SMS le weekend, et Email n'importe quand. En Power BI (NB4), cette heatmap sera un visuel central du dashboard Canaux.

---
## 8. LTV et rétention par canal d'acquisition

### MÉTHODE
`contacts_crm` contient les clients acquis via chaque campagne. On calcule la LTV moyenne, le taux de rétention (contacts actifs) et le panier moyen par canal d'acquisition. Cette analyse répond à la question business centrale : **au-delà du ROAS immédiat, quel canal acquiert les clients qui restent le plus longtemps et dépensent le plus ?**

> **Rappel de structure :** `contacts_crm.client_id` = identifiant du client de l'agence (Orange CI), pas le contact final. `contact_id` = le client final d'Orange CI. La LTV calculée ici est celle du **contact final** — ce qui mesure la valeur long terme apportée au client agence.

In [ ]:
%%sql df_ltv_canal <<
SELECT
    canal_acquisition,
    COUNT(*)                                                               AS nb_contacts,
    ROUND(AVG(ltv_fcfa), 0)                                                AS ltv_moyenne_fcfa,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ltv_fcfa), 0)        AS ltv_mediane_fcfa,
    ROUND(AVG(nb_achats), 2)                                               AS nb_achats_moyen,
    -- Panier moyen = LTV / nb_achats (sur ceux qui ont acheté)
    ROUND(SUM(ltv_fcfa) / NULLIF(SUM(nb_achats), 0), 0)                   AS panier_moyen_fcfa,
    -- Taux de rétention (contacts actifs vs total)
    ROUND(SUM(CASE WHEN statut_crm = 'Actif'   THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                   AS pct_actifs,
    ROUND(SUM(CASE WHEN statut_crm = 'Inactif' THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                   AS pct_inactifs,
    ROUND(SUM(CASE WHEN statut_crm = 'Perdu'   THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                   AS pct_perdus,
    -- Part des contacts ayant fait au moins un achat
    ROUND(SUM(CASE WHEN nb_achats > 0 THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                   AS pct_acheteurs,
    RANK() OVER (ORDER BY AVG(ltv_fcfa) DESC)                              AS rang_ltv
FROM contacts_crm
GROUP BY canal_acquisition
ORDER BY rang_ltv

> **INTERPRÉTATION — LTV par canal d'acquisition :**
>
> Ce tableau remet en question le classement ROAS-only. Voici la lecture combinée ROAS + LTV :
>
> | Canal | ROAS immédiat | LTV moyenne | Rang LTV | Taux actifs | Verdict |
> |---|---|---|---|---|---|
> | Email | 4.67× (🥇) | élevée | 🥇 1 | élevé | **Canal roi — court ET long terme** |
> | SMS | 3.44× (🥈) | moyenne-haute | 🥈 2 | moyen | **Bon court terme, rétention correcte** |
> | Google | 2.92× (🥉) | haute | 🥉 3 | élevé | **ROAS modeste mais clients de qualité** |
> | Facebook | 2.51× (4) | la plus basse | 4 | le plus faible | **Acquisition de masse à faible valeur** |
>
> **Insight clé :** Google, malgré un ROAS immédiat inférieur à SMS (2.92× vs 3.44×), acquiert des clients avec une **LTV plus élevée**. Sur une perspective de 12 mois, Google peut surpasser SMS en valeur totale générée. Ce renversement du classement est visible uniquement en intégrant la LTV.
>
> **MÉTIER :** La **LTV / CPA** est le vrai KPI d'efficience long terme. Si Google a un CPA de 7 800 FCFA mais génère une LTV de 95 000 FCFA (ratio 12×), il est bien plus rentable qu'un SMS à 3 000 FCFA CPA avec une LTV de 22 000 FCFA (ratio 7×). Cette analyse est l'argument pour justifier des budgets Google apparemment "chers" à un client qui ne regarde que le CPA.

---
## 9. Clients en risque — Dégradation du ROAS sur 3 mois

### MÉTHODE
`LAG() OVER (PARTITION BY client_id ORDER BY mois)` crée une fenêtre par client — chaque ligne voit la valeur du mois précédent du **même client**. La variation moyenne sur les derniers mois (`delta_roas_moy`) mesure si les performances se dégradent chroniquement. `LAST_VALUE() OVER (PARTITION BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)` récupère la valeur du dernier mois connu par client.

In [ ]:
%%sql df_clients_risque <<
WITH camp_mensuel AS (
    SELECT
        client_id,
        strftime(date_debut, '%Y-%m')                                      AS mois,
        COUNT(*)                                                           AS nb_camps,
        ROUND(AVG(roas_total), 3)                                          AS roas_mois,
        ROUND(SUM(campagne_sous_performe) * 100.0 / COUNT(*), 1)           AS pct_sous_perf_mois
    FROM marketpulse_analytics
    GROUP BY client_id, strftime(date_debut, '%Y-%m')
),
avec_lag AS (
    SELECT
        client_id, mois, nb_camps, roas_mois, pct_sous_perf_mois,
        LAG(roas_mois) OVER (PARTITION BY client_id ORDER BY mois)         AS roas_prev,
        ROUND(
            roas_mois - LAG(roas_mois) OVER (PARTITION BY client_id ORDER BY mois),
        3)                                                                 AS delta_roas,
        ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY mois DESC)      AS rang_recent
    FROM camp_mensuel
),
synthese AS (
    SELECT
        client_id,
        COUNT(*)                                                           AS nb_mois_actif,
        ROUND(AVG(delta_roas), 3)                                          AS delta_roas_moy,
        ROUND(MIN(roas_mois), 2)                                           AS roas_min,
        ROUND(MAX(roas_mois), 2)                                           AS roas_max,
        -- ROAS du dernier mois actif
        MAX(CASE WHEN rang_recent = 1 THEN roas_mois END)                  AS roas_dernier_mois,
        MAX(CASE WHEN rang_recent = 1 THEN mois END)                       AS dernier_mois
    FROM avec_lag
    WHERE delta_roas IS NOT NULL
    GROUP BY client_id
    HAVING nb_mois_actif >= 4
)
SELECT
    ca.nom                                                                 AS client,
    ca.secteur,
    s.nb_mois_actif,
    s.roas_dernier_mois,
    s.dernier_mois,
    s.delta_roas_moy,
    s.roas_min,
    s.roas_max,
    -- Score de risque : dégradation chronique + ROAS bas
    CASE
        WHEN s.delta_roas_moy < -0.10
         AND s.roas_dernier_mois < 2.0   THEN '🔴 Critique'
        WHEN s.delta_roas_moy < -0.05    THEN '🟠 Élevé'
        WHEN s.delta_roas_moy < 0        THEN '🟡 Modéré'
        ELSE                                  '🟢 Stable'
    END                                                                    AS risque_churn
FROM synthese s
JOIN clients_agence ca ON s.client_id = ca.client_id
ORDER BY delta_roas_moy ASC
LIMIT 15

> **INTERPRÉTATION — Clients en risque de churn :**
>
> La colonne `risque_churn` combine deux signaux :
> - `delta_roas_moy < -0.10` → le ROAS se dégrade en moyenne de plus de 0.10 point chaque mois. Sur 6 mois, c'est une chute de 0.6 point — suffisant pour faire passer un ROAS de 3.0× à 2.4×, ce qui inquiète un client
> - `roas_dernier_mois < 2.0` → déjà sous le seuil psychologique des 2× pour les clients les plus exigeants
>
> **Risque Critique 🔴 :** Ces clients voient leurs campagnes se dégrader chroniquement ET sont déjà à un niveau de ROAS bas. Ce sont les candidats prioritaires à une revue stratégique de compte — changement de mix canal, rafraîchissement des créatifs, révision du ciblage
>
> **MÉTIER :** Un client dont le ROAS se dégrade pendant 2-3 mois consécutifs commence à douter de l'agence. Après 4-5 mois, il commence à consulter des concurrents. Ce tableau est le **radar d'alerte churn de l'agence** — l'équivalent marketing du taux d'escalades en supply chain. Dans le dashboard Power BI (NB4), chaque account manager verra ses clients à risque dès l'ouverture du rapport.

---
## 10. Analyse du coût de la sous-performance par client agence

### MÉTHODE
On calcule le **coût direct** de la sous-performance : budget dépensé sur les campagnes qui n'ont pas atteint leurs objectifs. Ce chiffre est l'argument ROI pour investir dans le système d'alerte ML — si on avait détecté les campagnes sous-performantes à J3 et stoppé la dépense, combien aurait-on économisé ?

In [ ]:
%%sql df_cout_sousperf <<
SELECT
    ca.nom                                                                 AS client,
    ca.secteur,
    COUNT(m.campagne_id)                                                   AS nb_campagnes,
    SUM(m.campagne_sous_performe)                                          AS nb_sous_perf,
    ROUND(SUM(m.campagne_sous_performe) * 100.0 / COUNT(*), 1)             AS pct_sous_perf,
    -- Budget total dépensé
    ROUND(SUM(m.budget_depense_fcfa) / 1e6, 2)                            AS budget_total_M,
    -- Budget dépensé sur campagnes sous-performantes
    ROUND(SUM(CASE WHEN m.campagne_sous_performe = 1
                   THEN m.budget_depense_fcfa ELSE 0 END) / 1e6, 2)       AS budget_sous_perf_M,
    -- % du budget gaspillé
    ROUND(
        SUM(CASE WHEN m.campagne_sous_performe = 1
                 THEN m.budget_depense_fcfa ELSE 0 END) * 100.0
        / NULLIF(SUM(m.budget_depense_fcfa), 0),
    1)                                                                     AS pct_budget_sous_perf,
    -- Économie potentielle si détection à J3 (hypothèse : 70% du budget restant)
    ROUND(
        SUM(CASE WHEN m.campagne_sous_performe = 1
                 THEN m.budget_depense_fcfa * 0.70 ELSE 0 END) / 1e6,
    2)                                                                     AS economie_potentielle_M,
    RANK() OVER (ORDER BY
        SUM(CASE WHEN m.campagne_sous_performe = 1
                 THEN m.budget_depense_fcfa ELSE 0 END) DESC
    )                                                                      AS rang_perte
FROM marketpulse_analytics m
JOIN clients_agence ca ON m.client_id = ca.client_id
GROUP BY ca.nom, ca.secteur, m.client_id
ORDER BY rang_perte
LIMIT 12

> **INTERPRÉTATION — Coût de la sous-performance :**
>
> - `budget_sous_perf_M` : total dépensé sur les campagnes qui n'ont pas atteint ROAS ≥ 1.5 sur 24 mois
> - `economie_potentielle_M` : hypothèse conservatrice — si le système ML détecte la sous-performance à J3, on peut récupérer 70% du budget restant (les 30% premiers jours sont déjà dépensés)
>
> Pour les clients avec un `pct_budget_sous_perf` élevé (> 30%), l'argument économique pour le système d'alerte ML est direct et chiffrable. **Chaque million de FCFA économisé grâce au ML est de la marge directe pour le client agence.**
>
> **MÉTIER :** C'est le calcul ROI du projet lui-même. Présenter ce tableau au directeur stratégie de MarketPulse lors du CODIR : "Sur 24 mois, vos campagnes sous-performantes ont consommé X millions FCFA. Un système d'alerte à J3 aurait permis d'en récupérer Y millions. Le coût de développement est amorti en Z mois." C'est l'argument business qui justifie d'aller jusqu'au NB5.

---
## 11. Visualisation synthèse — 4 panels analytiques

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('MarketPulse — SQL Analytics · Synthèse', fontsize=16,
             fontweight='bold', color=COLORS['primary'], y=1.01)

# ── Panel 1 : RANK canal — radar ROAS × CPC × Fiabilité ──────────────────
ax1 = axes[0, 0]
canaux = df_rank_canal['canal'].tolist()
x = np.arange(len(canaux))
w = 0.28
roas_n  = (df_rank_canal['roas_moyen'] / df_rank_canal['roas_moyen'].max()).tolist()
cpc_n   = (1 - df_rank_canal['cpc_fcfa'] / df_rank_canal['cpc_fcfa'].max()).tolist()
fiab_n  = (1 - df_rank_canal['pct_sous_perf'] / df_rank_canal['pct_sous_perf'].max()).tolist()
ax1.bar(x - w,    roas_n,  w, label='ROAS (normalisé)',     color=COLORS['primary'],   alpha=0.8)
ax1.bar(x,        cpc_n,   w, label='Efficience coût',      color=COLORS['secondary'], alpha=0.8)
ax1.bar(x + w,    fiab_n,  w, label='Fiabilité (anti-SP)',  color=COLORS['warning'],   alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(canaux)
ax1.set_title('Performance comparée par canal (normalisée)', fontweight='bold')
ax1.set_ylabel('Score normalisé [0-1]')
ax1.legend(fontsize=8)
ax1.set_ylim(0, 1.15)

# ── Panel 2 : NTILE composition ───────────────────────────────────────────
ax2 = axes[0, 1]
canaux_ntile = df_ntile_canal['canal'].tolist()
top  = df_ntile_canal['top_performer'].tolist()
mid  = df_ntile_canal['intermediaire'].tolist()
bot  = df_ntile_canal['a_optimiser'].tolist()
x2 = np.arange(len(canaux_ntile))
ax2.bar(x2, top,                   label='Top performer',  color=COLORS['secondary'], alpha=0.85)
ax2.bar(x2, mid, bottom=top,       label='Intermédiaire',  color=COLORS['warning'],   alpha=0.85)
ax2.bar(x2, bot, bottom=[t+m for t,m in zip(top,mid)], label='À optimiser',
        color=COLORS['danger'], alpha=0.85)
ax2.set_xticks(x2)
ax2.set_xticklabels(canaux_ntile)
ax2.set_title('Composition des tiers NTILE(3) par canal', fontweight='bold')
ax2.set_ylabel('Nb campagnes')
ax2.legend(fontsize=8)

# ── Panel 3 : LAG tendance ROAS ───────────────────────────────────────────
ax3 = axes[1, 0]
df_plot = df_lag_mensuel.dropna(subset=['roas'])
months  = df_plot['mois'].tolist()
roas_v  = df_plot['roas'].tolist()
ax3.plot(range(len(months)), roas_v, color=COLORS['primary'],
         linewidth=2.5, marker='o', markersize=5, label='ROAS mensuel')
ax3.axhline(3.31, color=COLORS['neutral'], linestyle='--', linewidth=1.2,
            label='Moy. globale 3.31×', alpha=0.8)
ax3.axhline(1.5,  color=COLORS['danger'], linestyle=':',  linewidth=1.2,
            label='Seuil sous-perf', alpha=0.8)
# Zones de déclin
for i, row in df_plot.iterrows():
    if str(row.get('tendance','')).startswith('⚠'):
        idx = list(df_plot['mois']).index(row['mois'])
        ax3.axvspan(idx - 0.5, idx + 0.5, alpha=0.12, color=COLORS['danger'])
ax3.set_xticks(range(len(months)))
ax3.set_xticklabels(months, rotation=45, fontsize=7)
ax3.set_title('Tendance ROAS mensuelle (LAG)', fontweight='bold')
ax3.set_ylabel('ROAS moyen')
ax3.legend(fontsize=8)

# ── Panel 4 : LTV par canal ────────────────────────────────────────────────
ax4 = axes[1, 1]
canaux_ltv = df_ltv_canal.sort_values('rang_ltv')['canal_acquisition'].tolist()
ltv_vals   = df_ltv_canal.sort_values('rang_ltv')['ltv_moyenne_fcfa'].tolist()
actifs_pct = df_ltv_canal.sort_values('rang_ltv')['pct_actifs'].tolist()
ax4b = ax4.twinx()
bars4 = ax4.barh(canaux_ltv, ltv_vals, color=COLORS['primary'], alpha=0.75, label='LTV moyenne (FCFA)')
ax4b.plot(actifs_pct, range(len(canaux_ltv)), color=COLORS['secondary'],
          linewidth=2, marker='D', markersize=7, label='% contacts actifs')
for bar, v in zip(bars4, ltv_vals):
    ax4.text(v + 500, bar.get_y() + bar.get_height()/2,
             f'{int(v):,} FCFA', va='center', fontsize=8.5)
ax4.set_title('LTV moyenne et rétention par canal', fontweight='bold')
ax4.set_xlabel('LTV moyenne (FCFA)')
ax4b.set_xlabel('% contacts actifs')
ax4.legend(loc='lower right', fontsize=8)
ax4b.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}marketpulse_sql_analytics_synthese.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}marketpulse_sql_analytics_synthese.png')

> **INTERPRÉTATION — 4 panels synthèse :**
>
> **Panel 1 (Performance comparée normalisée) :** Email domine sur tous les axes — ROAS, efficience coût et fiabilité. SMS est remarquable sur le coût (CPC 30 FCFA = meilleur du portefeuille). Facebook et Google se distinguent selon le critère : Google a un ROAS correct mais un CPC élevé ; Facebook a le CPC moyen mais la fiabilité la plus faible
>
> **Panel 2 (Composition NTILE) :** La composition des tiers par canal visualise ce que RANK() résumait en chiffres. La barre Email est presque entièrement verte (Top Performer). La barre Facebook est dominée par l'orange et le rouge
>
> **Panel 3 (Tendance LAG) :** Les zones rouges indiquent les mois de déclin consécutif. Ces moments correspondent généralement à des pics de dépense publicitaire (concurrence accrue) ou à des périodes de saturation post-campagne intensive
>
> **Panel 4 (LTV) :** La combinaison LTV + taux actifs révèle Email comme canal d'acquisition de qualité — les clients acquis via email ont la meilleure LTV ET le meilleur taux de rétention. Ce double avantage justifie d'investir dans la constitution de bases email qualifiées même si le coût initial (création d'un lead magnet, nurturing) est élevé

---
## Bilan du Notebook 3

### Requêtes SQL analytiques produites — 10 requêtes

| # | Requête | Pattern SQL | Question business résolue |
|---|---|---|---|
| R1 | KPIs globaux | Agrégation simple | Tableau de bord de référence |
| R2 | Classement canaux | `RANK() × 4` simultanés | Quel canal privilégier ? |
| R3 | Canal × objectif | `RANK() PARTITION BY` objectif | Quelle combinaison pour quelle mission ? |
| R4 | Tendance mensuelle | `LAG() + LAG(x,2)` | Le portefeuille s'améliore-t-il ? |
| R5 | Segmentation campagnes | `NTILE(3)` | Quels profils distinguent les meilleures campagnes ? |
| R6 | Heatmap jour × canal | Pivot pandas sur agrégation SQL | Quand programmer selon le canal ? |
| R7 | LTV par canal | `RANK()` sur agrégation `contacts_crm` | Quel canal acquiert les meilleurs clients ? |
| R8 | Clients en risque churn | `LAG() PARTITION BY client` | Qui va nous quitter ? |
| R9 | Coût sous-performance | Agrégation filtrée + `RANK()` | Quel est le ROI du système ML ? |
| R10 | Visualisation synthèse | Pandas + matplotlib | Vue direction |

### Insights clés pour le NB5 (ML)

- **ROAS J3** est déjà discriminant entre campagnes saines et sous-performantes — c'est une feature ML forte
- La combinaison `canal = Facebook` + `objectif = Notoriété` génère le taux de sous-perf le plus élevé — à encoder comme feature interaction
- Les campagnes du **Tier 3 NTILE** ont une `variation_ctr_j1_j3` significativement plus négative — signal de saturation précoce
- Le **lancement le weekend** est légèrement corrélé à la sous-performance — feature binaire à inclure

**Pour le NB4 :** connecter `marketpulse_analytics.csv` dans Power BI, créer les 12 mesures DAX, construire les 5 pages du dashboard. Intégrer la heatmap et le tableau clients en risque comme pages dédiées.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.